# Telecom Churn Prediction - Exploratory Data Analysis

This notebook explores the telecom customer data to understand patterns and prepare for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.config import DATA_DIR

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Validate Data

In [ ]:
# Load data
loader = DataLoader()
df = loader.load_data()

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)

## 2. Identify High-Value Customers

In [ ]:
# Identify high-value customers
df = loader.identify_high_value_customers(df)

# Distribution of high-value customers
high_value_counts = df['high_value'].value_counts()
print("High-value customer distribution:")
print(high_value_counts)
print(f"\nPercentage of high-value customers: {high_value_counts[1] / len(df) * 100:.1f}%")

In [ ]:
# Visualize recharge distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df['avg_recharge_good_phase'].dropna(), bins=50, edgecolor='black')
plt.xlabel('Average Recharge Amount')
plt.ylabel('Frequency')
plt.title('Distribution of Average Recharge (Good Phase)')

plt.subplot(1, 2, 2)
df.boxplot(column='avg_recharge_good_phase', by='high_value')
plt.title('Recharge Amount by Customer Value')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 3. Define and Analyze Churn

In [ ]:
# Define churn
df = loader.define_churn(df)

# Churn rate
churn_rate = df['churned'].mean()
print(f"Overall churn rate: {churn_rate:.2%}")

# Churn by high-value status
churn_by_value = df.groupby('high_value')['churned'].agg(['count', 'sum', 'mean'])
churn_by_value.columns = ['total_customers', 'churned_customers', 'churn_rate']
print("\nChurn by customer value:")
print(churn_by_value)

In [ ]:
# Visualize churn
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
churn_counts = df['churned'].value_counts()
plt.pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', startangle=90)
plt.title('Overall Churn Distribution')

plt.subplot(1, 2, 2)
churn_by_value['churn_rate'].plot(kind='bar')
plt.xlabel('High Value Customer')
plt.ylabel('Churn Rate')
plt.title('Churn Rate by Customer Value')
plt.xticks([0, 1], ['Regular', 'High-Value'], rotation=0)

plt.tight_layout()
plt.show()

## 4. Usage Pattern Analysis

In [ ]:
# Filter to high-value customers for detailed analysis
high_value_df = loader.get_high_value_subset(df)

# Usage trends
usage_cols = [col for col in high_value_df.columns if 'total_calls' in col]
if usage_cols:
    usage_by_churn = high_value_df.groupby('churned')[usage_cols].mean()
    
    plt.figure(figsize=(12, 4))
    usage_by_churn.T.plot(kind='bar')
    plt.xlabel('Month')
    plt.ylabel('Average Calls')
    plt.title('Call Usage Trends by Churn Status (High-Value Customers)')
    plt.legend(['No Churn', 'Churn'])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation with churn
numeric_df = high_value_df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['churned'].sort_values(ascending=False)

print("Top correlations with churn:")
print(correlations.head(10))
print("\nTop negative correlations:")
print(correlations.tail(10))

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
print("=== DATA SUMMARY ===")
print(f"Total customers: {df['customer_id'].nunique()}")
print(f"High-value customers: {high_value_df['customer_id'].nunique()}")
print(f"Overall churn rate: {df['churned'].mean():.2%}")
print(f"High-value churn rate: {high_value_df['churned'].mean():.2%}")